In [1]:
from fedbiomed.common.training_plans import TorchTrainingPlan
from fedbiomed.common.datamanager import DataManager
from fedbiomed.common.dataset import MedicalFolderDataset
from fedbiomed.researcher.federated_workflows import Experiment
from fedbiomed.researcher.aggregators.fedavg import FedAverage
 
import torch
import torch.nn as nn
from torch.optim import AdamW
from unet import UNet

W0907 15:49:49.660000 95034 torch/distributed/elastic/multiprocessing/redirects.py:29] NOTE: Redirects are currently not supported in Windows or MacOs.


2026-09-07 15:49:49,786 fedbiomed INFO - Syslog configuration is disabled or not found in configuration. Disabling syslog logging.

In [2]:
# --------------------------------------------------------------------------- #
#  Training plan (training only)
# --------------------------------------------------------------------------- #
class UNetValidationPlan(TorchTrainingPlan):
 
    def init_model(self, model_args):
        return self.Net(model_args)
 
    def init_optimizer(self, optimizer_args):
        # lr = 0.0 -> the optimizer is a no-op. AdamW uses decoupled weight
        # decay (p -= lr * wd * p), so lr=0 also disables weight decay.
        return AdamW(self.model().parameters(),
                     lr=optimizer_args.get('lr', 0.0),
                     weight_decay=0.0)
 
    def init_dependencies(self):
        return [
            "from monai.transforms import (Compose, NormalizeIntensity, "
            "EnsureChannelFirst, Resize, AsDiscrete)",
            "import torch",
            "import torch.nn as nn",
            "import torch.nn.functional as F",
            "from fedbiomed.common.dataset import MedicalFolderDataset",
            "import numpy as np",
            "from torch.optim import AdamW",
            "from unet import UNet",
        ]
 
    class Net(nn.Module):
        def __init__(self, model_args: dict = {}):
            super().__init__()
            self.CHANNELS_DIMENSION = 1
            self.unet = UNet(
                in_channels=model_args.get('in_channels', 1),
                out_classes=model_args.get('out_classes', 2),
                dimensions=model_args.get('dimensions', 2),
                num_encoding_blocks=model_args.get('num_encoding_blocks', 5),
                out_channels_first_layer=model_args.get('out_channels_first_layer', 64),
                normalization=model_args.get('normalization', None),
                pooling_type=model_args.get('pooling_type', 'max'),
                upsampling_type=model_args.get('upsampling_type', 'conv'),
                preactivation=model_args.get('preactivation', False),
                residual=model_args.get('residual', False),
                padding=model_args.get('padding', 0),
                padding_mode=model_args.get('padding_mode', 'zeros'),
                activation=model_args.get('activation', 'ReLU'),
                initial_dilation=model_args.get('initial_dilation', None),
                dropout=model_args.get('dropout', 0),
                monte_carlo_dropout=model_args.get('monte_carlo_dropout', 0),
            )
 
        def forward(self, x):
            x = self.unet.forward(x)
            return F.softmax(x, dim=self.CHANNELS_DIMENSION)
 
    # Dice loss evaluation --------------------------------------------------------- #
    @staticmethod
    def get_dice_score(output, target, epsilon=1e-9):
        """Per-sample, per-class Dice score. Returns a (batch, n_classes) tensor."""
        SPATIAL_DIMENSIONS = 2, 3, 4
        p0, g0 = output, target
        p1, g1 = 1 - p0, 1 - g0
        tp = (p0 * g0).sum(dim=SPATIAL_DIMENSIONS)
        fp = (p0 * g1).sum(dim=SPATIAL_DIMENSIONS)
        fn = (p1 * g0).sum(dim=SPATIAL_DIMENSIONS)
        return (2 * tp) / (2 * tp + fp + fn + epsilon)
 
    @staticmethod
    def get_dice_loss(output, target, epsilon=1e-9):
        return 1. - UNetValidationPlan.get_dice_score(output, target, epsilon)
 
    # Data Loading  -------------------------------------------------------------- #
    def training_data(self):
        common_shape = (48, 60, 48)
 
        image_transform = Compose([
            EnsureChannelFirst(channel_dim="no_channel"),
            Resize(common_shape),
            NormalizeIntensity(),
        ])
        target_transform = Compose([
            EnsureChannelFirst(channel_dim="no_channel"),
            Resize(common_shape),
            AsDiscrete(to_onehot=2),
        ])
 
        # shuffle=False: nothing is learned, and a deterministic order makes
        # per-sample results reproducible from one run to another.
        loader_arguments = {'shuffle': False}
 
        mf = MedicalFolderDataset(
            data_modalities='T1',
            target_modalities='label',
            transform={'T1': image_transform},
            target_transform={'label': target_transform},
        )
        return DataManager(mf, **loader_arguments)
 
    # Training ----------------------------------------------------------------- #
    def training_step(self, data, target):
        """
        With test_ratio=1.0 this method is never called (empty train loader).
        It is kept gradient-safe so the plan still works if you fall back to
        test_ratio < 1.0. eval() freezes BatchNorm running statistics.
        """

        self.model().eval()
        output = self.model().forward(data['T1'])
        return UNetValidationPlan.get_dice_loss(output, target['label']).mean()


In [3]:
model_args = {
    'in_channels': 1,
    'out_classes': 2,
    'dimensions': 3,
    'num_encoding_blocks': 3,
    'out_channels_first_layer': 8,
    'normalization': 'batch',
    'upsampling_type': 'linear',
    'padding': True,
    'activation': 'PReLU',
}
 
training_args = {
    'loader_args': {'batch_size': 16},
    'optimizer_args': {'lr': 0.1},
    'epochs': 5,
    'dry_run': False,
    'log_interval': 1,
    'random_seed': 1234,
}
 
tags = ['brain-segmentation']   # or 'ixi-holdout' if you deployed the holdout folders
num_rounds = 10      # 1 round -> validation before and after (identical here)
 

# --------------------------------------------------------------------------- #
#  Build, load initial weights, run
# --------------------------------------------------------------------------- #
exp = Experiment(
    tags=tags,
    model_args=model_args,
    training_plan_class=UNetValidationPlan,
    training_args=training_args,
    round_limit=num_rounds,
    aggregator=FedAverage(),
    tensorboard=True,
)

2026-09-07 15:49:49,853 fedbiomed INFO - Starting researcher service...

2026-09-07 15:49:49,854 fedbiomed INFO - Waiting 3s for nodes to connect...

2026-09-07 15:49:52,861 fedbiomed INFO - Updating training data. This action will update FederatedDataset, and the nodes that will participate to the experiment.

2026-09-07 15:49:52,889 fedbiomed INFO - Node selected for training -> Default Node Name
Node ID is -> NODE_2a6199f3-fce8-4e5a-b878-1e2c08b2c3d1

2026-09-07 15:49:52,890 fedbiomed INFO - Node selected for training -> Default Node Name
Node ID is -> NODE_b6662ab9-8029-4995-bfa0-af3a5b68a18f

Secure RNG turned off. This is perfectly fine for experimentation as it allows for much faster training performance, but remember to turn it on and retrain one last time before production with ``secure_mode`` turned on.


In [ ]:
exp.run()

2026-09-07 15:50:03,828 fedbiomed INFO - Sampled nodes in round 0 ['NODE_2a6199f3-fce8-4e5a-b878-1e2c08b2c3d1', 'NODE_b6662ab9-8029-4995-bfa0-af3a5b68a18f']

2026-09-07 15:50:03,833 fedbiomed INFO - Sending request 
					 To: NODE_2a6199f3-fce8-4e5a-b878-1e2c08b2c3d1 
					 Request: : TRAIN
 -----------------------------------------------------------------

2026-09-07 15:50:03,833 fedbiomed INFO - Sending request 
					 To: NODE_b6662ab9-8029-4995-bfa0-af3a5b68a18f 
					 Request: : TRAIN
 -----------------------------------------------------------------

2026-09-07 15:50:51,314 fedbiomed INFO - TRAINING 
					 NODE_ID: NODE_2a6199f3-fce8-4e5a-b878-1e2c08b2c3d1 
					 Node Name: Default Node Name 
					 Round 1 Epoch: 1 | Iteration: 1/10 (10%) | Samples: 16/160
 					 Loss: 0.557708 
					 ---------

2026-09-07 15:51:09,671 fedbiomed INFO - TRAINING 
					 NODE_ID: NODE_2a6199f3-fce8-4e5a-b878-1e2c08b2c3d1 
					 Node Name: Default Node Name 
					 Round 1 Epoch: 1 | Iteration: 2/10 (20%) | Samples: 32/160
 					 Loss: 0.564461 
					 ---------

...

2026-09-07 16:10:00,111 fedbiomed INFO - TRAINING 
					 NODE_ID: NODE_2a6199f3-fce8-4e5a-b878-1e2c08b2c3d1 
					 Node Name: Default Node Name 
					 Round 2 Epoch: 1 | Iteration: 9/10 (90%) | Samples: 144/160
 					 Loss: 0.564016 
					 ---------

2026-09-07 16:10:15,394 fedbiomed INFO - TRAINING 
					 NODE_ID: NODE_b6662ab9-8029-4995-bfa0-af3a5b68a18f 
					 Node Name: Default Node Name 
					 Round 2 Epoch: 1 | Iteration: 11/18 (61%) | Samples: 176/288
 					 Loss: 0.564040 
					 ---------

2026-09-07 16:10:17,357 fedbiomed INFO - TRAINING 
					 NODE_ID: NODE_2a6199f3-fce8-4e5a-b878-1e2c08b2c3d1 
					 Node Name: Default Node Name 
					 Round 2 Epoch: 1 | Iteration: 10/10 (100%) | Samples: 159/159
 					 Loss: 0.564766 
					 ---------

2026-09-07 16:10:26,520 fedbiomed INFO - TRAINING 
					 NODE_ID: NODE_b6662ab9-8029-4995-bfa0-af3a5b68a18f 
					 Node Name: Default Node Name 
					 Round 2 Epoch: 1 | Iteration: 12/18 (67%) | Samples: 192/288
 					 Loss: 0.564395 
					 ---------

2026-09-07 16:10:28,797 fedbiomed INFO - TRAINING 
					 NODE_ID: NODE_2a6199f3-fce8-4e5a-b878-1e2c08b2c3d1 
					 Node Name: Default Node Name 
					 Round 2 Epoch: 2 | Iteration: 1/10 (10%) | Samples: 16/160
 					 Loss: 0.564215 
					 ---------

In [6]:
exp.training_plan().export_model('./brain-segmentation-trained-model.pt')